In [1]:
# Imports
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.linear_model import SGDClassifier, LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import accuracy_score, classification_report
import joblib

## 1. Load Preprocessed Data

In [2]:
# Paths
ROOT = Path.cwd().parents[0]
DATA = ROOT / "data"
PROCESSED = DATA / "processed"
SUBMISSION = DATA / "submission_linear"
SUBMISSION.mkdir(exist_ok=True)

# Load session-normalized data from preprocessing notebook
X_train = pd.read_csv(PROCESSED / "X_train_session_norm.csv").values
X_test = pd.read_csv(PROCESSED / "X_test_session_norm.csv").values
y_train = pd.read_csv(PROCESSED / "y_train.csv")['TRIAL_TYPE'].values
sessions_train = pd.read_csv(PROCESSED / "sessions_train.csv")['session_id'].values
sessions_test = pd.read_csv(PROCESSED / "sessions_test.csv")['session_id'].values

# Load feature names
with open(PROCESSED / "feature_names.txt") as f:
    feature_names = f.read().strip().split("\n")

print(f"Train: {X_train.shape}")
print(f"Test: {X_test.shape}")
print(f"Classes: {np.unique(y_train)}")
print(f"Sessions: {np.unique(sessions_train)}")

Train: (4407, 1350)
Test: (2292, 1350)
Classes: ['GO W+ lick' 'GO W+ nolick' 'GO W- lick' 'GO W- nolick' 'NOGO W+ lick'
 'NOGO W+ nolick' 'NOGO W- lick' 'NOGO W- nolick' 'no tone W+ lick'
 'no tone W+ nolick']
Sessions: ['PG082_20221113' 'PG082_20221114' 'PG083_20221125' 'PG083_20221129'
 'PG084_20221206' 'PG084_20221207' 'PG085_20221213' 'PG085_20221214']


## 2. LOSO Evaluation Function

In [3]:
def loso_evaluate(X, y, sessions, model_class, **model_params):
    """
    Leave-One-Session-Out cross-validation.
    
    Returns:
    --------
    mean_acc : float
    std_acc : float
    per_session : dict mapping session -> accuracy
    """
    unique_sessions = np.unique(sessions)
    results = {}
    
    for test_session in unique_sessions:
        train_mask = sessions != test_session
        test_mask = sessions == test_session
        
        model = model_class(**model_params)
        model.fit(X[train_mask], y[train_mask])
        pred = model.predict(X[test_mask])
        acc = accuracy_score(y[test_mask], pred)
        results[test_session] = acc
    
    accs = list(results.values())
    return np.mean(accs), np.std(accs), results

## 3. Evaluate Linear Models

In [4]:
# Test multiple linear models
print("="*60)
print("LINEAR MODEL EVALUATION (LOSO)")
print("="*60)

models = {
    'Logistic Regression (L2)': (LogisticRegression, {'C': 1.0, 'max_iter': 1000, 'n_jobs': -1, 'random_state': 42}),
    'Linear SVM (SGD)': (SGDClassifier, {'loss': 'hinge', 'alpha': 0.0001, 'max_iter': 1000, 'random_state': 42, 'n_jobs': -1}),
    'LDA': (LinearDiscriminantAnalysis, {}),
    'LDA (shrinkage)': (LinearDiscriminantAnalysis, {'solver': 'lsqr', 'shrinkage': 'auto'}),
}

results_summary = {}

for name, (model_class, params) in models.items():
    mean_acc, std_acc, per_session = loso_evaluate(X_train, y_train, sessions_train, model_class, **params)
    results_summary[name] = (mean_acc, std_acc)
    print(f"\n{name}:")
    print(f"  LOSO Accuracy: {mean_acc:.1%} ± {std_acc:.1%}")
    for sess, acc in per_session.items():
        print(f"    {sess}: {acc:.1%}")

# Summary
print("\n" + "="*60)
print("SUMMARY")
print("="*60)
for name, (mean_acc, std_acc) in sorted(results_summary.items(), key=lambda x: -x[1][0]):
    print(f"{name:30s}: {mean_acc:.1%} ± {std_acc:.1%}")

LINEAR MODEL EVALUATION (LOSO)

Logistic Regression (L2):
  LOSO Accuracy: 42.5% ± 4.6%
    PG082_20221113: 43.8%
    PG082_20221114: 36.2%
    PG083_20221125: 50.8%
    PG083_20221129: 41.5%
    PG084_20221206: 43.5%
    PG084_20221207: 44.8%
    PG085_20221213: 35.6%
    PG085_20221214: 43.5%

Linear SVM (SGD):
  LOSO Accuracy: 43.0% ± 4.4%
    PG082_20221113: 43.7%
    PG082_20221114: 42.0%
    PG083_20221125: 53.1%
    PG083_20221129: 41.5%
    PG084_20221206: 36.8%
    PG084_20221207: 43.4%
    PG085_20221213: 39.6%
    PG085_20221214: 43.7%

LDA:
  LOSO Accuracy: 39.3% ± 3.8%
    PG082_20221113: 39.0%
    PG082_20221114: 36.5%
    PG083_20221125: 43.6%
    PG083_20221129: 36.6%
    PG084_20221206: 41.7%
    PG084_20221207: 43.2%
    PG085_20221213: 31.9%
    PG085_20221214: 41.5%


c:\Users\milor\anaconda3\envs\MLCourse\lib\site-packages\sklearn\covariance\_shrunk_covariance.py:349: UserWarning: Only one sample available. You may want to reshape your data array
  warnings.warn(
c:\Users\milor\anaconda3\envs\MLCourse\lib\site-packages\sklearn\covariance\_empirical_covariance.py:102: UserWarning: Only one sample available. You may want to reshape your data array
  warnings.warn(



LDA (shrinkage):
  LOSO Accuracy: 48.3% ± 4.7%
    PG082_20221113: 49.1%
    PG082_20221114: 45.6%
    PG083_20221125: 54.9%
    PG083_20221129: 47.0%
    PG084_20221206: 47.0%
    PG084_20221207: 51.1%
    PG085_20221213: 38.8%
    PG085_20221214: 53.2%

SUMMARY
LDA (shrinkage)               : 48.3% ± 4.7%
Linear SVM (SGD)              : 43.0% ± 4.4%
Logistic Regression (L2)      : 42.5% ± 4.6%
LDA                           : 39.3% ± 3.8%


## 4. Train Best Linear Model & Generate Submission

In [5]:
# Select best model (LDA with shrinkage typically performs best)
best_model_name = max(results_summary.items(), key=lambda x: x[1][0])[0]
print(f"Best linear model: {best_model_name}")

# Train on full training set
if 'shrinkage' in best_model_name.lower():
    final_model = LinearDiscriminantAnalysis(solver='lsqr', shrinkage='auto')
elif 'LDA' in best_model_name:
    final_model = LinearDiscriminantAnalysis()
elif 'SVM' in best_model_name:
    final_model = SGDClassifier(loss='hinge', alpha=0.0001, max_iter=1000, random_state=42)
else:
    final_model = LogisticRegression(C=1.0, max_iter=1000, n_jobs=-1, random_state=42)

final_model.fit(X_train, y_train)
print(f"Model trained on {len(X_train)} samples")

# Generate predictions
test_predictions = final_model.predict(X_test)
print(f"\nPrediction distribution:")
unique, counts = np.unique(test_predictions, return_counts=True)
for cls, cnt in zip(unique, counts):
    print(f"  {cls}: {cnt}")

Best linear model: LDA (shrinkage)
Model trained on 4407 samples

Prediction distribution:
  GO W+ lick: 396
  GO W+ nolick: 46
  GO W- lick: 8
  GO W- nolick: 468
  NOGO W+ lick: 27
  NOGO W+ nolick: 422
  NOGO W- nolick: 523
  no tone W+ lick: 22
  no tone W+ nolick: 380


In [7]:
# Create and save submission (matching sample_submission.csv format)
submission = pd.DataFrame({
    'ID': range(1, len(test_predictions) + 1),  # 1-based indexing
    'TRIAL_TYPE': test_predictions
})

# Save submission
submission_path = SUBMISSION / "linear_submission.csv"
submission.to_csv(submission_path, index=False)
print(f"\nSubmission saved to: {submission_path}")
print(f"Shape: {submission.shape}")

# Save model
model_path = SUBMISSION / "linear_model.joblib"
joblib.dump(final_model, model_path)
print(f"Model saved to: {model_path}")

# Display first few predictions
print("\nFirst 10 predictions:")
print(submission.head(10))


Submission saved to: c:\Users\milor\Projects\ML-BIO-322\BIO-322\data\submission_linear\linear_submission.csv
Shape: (2292, 2)
Model saved to: c:\Users\milor\Projects\ML-BIO-322\BIO-322\data\submission_linear\linear_model.joblib

First 10 predictions:
   ID       TRIAL_TYPE
0   1     GO W- nolick
1   2       GO W+ lick
2   3       GO W- lick
3   4     NOGO W+ lick
4   5       GO W+ lick
5   6   NOGO W+ nolick
6   7   NOGO W- nolick
7   8  no tone W+ lick
8   9   NOGO W+ nolick
9  10   NOGO W+ nolick


## Summary

Linear models on session-normalized data achieve ~42-48% LOSO accuracy.

This is significantly lower than non-linear models (Random Forest: ~58%), suggesting that
the decision boundaries in neural space are non-linear.

**Best linear model:** LDA with shrinkage (~48%)